In [1]:
import torch
import einops

In [2]:
def vector_gather(vectors, indices):
    """
    Gathers (batched) vectors according to indices.
    Arguments:
        vectors: Tensor[N, L, D]
        indices: Tensor[N, K] or Tensor[N]
    Returns:
        Tensor[N, K, D] or Tensor[N, D]
    """
    N, L, D = vectors.shape
    squeeze = False
    if indices.ndim == 1:
        squeeze = True
        indices = indices.unsqueeze(-1)
    N2, K = indices.shape
    assert N == N2
    indices = einops.repeat(indices, "N K -> N K D", D=D)
    out = torch.gather(vectors, dim=1, index=indices)
    if squeeze:
        out = out.squeeze(1)
    return out

In [3]:
def dag_loss(targets, transition_matrix, emission_probs, bos_idx=0):
    batch_size, m = targets.shape
    _, l, vocab_size = emission_probs.shape
    dp = torch.ones((batch_size, m, l))
    dp[dp == 1] = -float('inf')
    initial_probs = torch.gather(emission_probs, dim=2, index=targets[:, 0].unsqueeze(1).unsqueeze(2))
    dp[:, 0, 0] = initial_probs.squeeze(2).squeeze(1)
    # assumes that transition_matrix and emission_probs are already in log space
    # also we need to tranpose emission_probs so it is vocab_size x l
    # so the vector gather works
    emission_probs = emission_probs.transpose(1, 2)
    for i in range(1, m):
        dp[:, i, :] = vector_gather(emission_probs, targets[:, i]) + (torch.logsumexp(dp[:, i-1, :].unsqueeze(1).transpose(1, 2) + transition_matrix, dim=1))
    return dp

In [4]:
def fix_probs(logprobs, mask):
    # assumes probs is already in log space
    # and is a square matrix
    # updates probs so that the sum of each row is 1
    # and any available probability mass is 
    # distributed evenly among the non-masked entries
    batch_size, l, _ = logprobs.shape

    # any part of the mask where the value is not 0, we mask out
    logprobs = logprobs.masked_fill(mask != 0, float('-inf'))
    probsmatrix = torch.exp(logprobs)
    remaining = torch.sum(probsmatrix, dim=2)
    remaining = 1 - remaining
    probnonzero = torch.sum(mask == 0, dim=-1)
    remaining = remaining / probnonzero
    probsmatrix = probsmatrix + remaining.unsqueeze(2)

    # at this point, it is mostly correct, however if there 
    # are rows where the number of non zeros (probnonzero)
    # is 0, then we get infinities at those positions, which
    # are obviously wrong, so we keep only positions
    # where we haven't masked out
    probsmatrix = probsmatrix.masked_fill(mask != 0, 0)
    logprobs = torch.log(probsmatrix)
    return logprobs

In [5]:
m = 7
l = 9
padded_l = l + 3
vocab_size = 9

# the 0s after the 8 are padding
target = [0, 1, 2, 3, 4, 5, 8, 0, 0, 0, 0, 0]
target = torch.tensor(target)

In [6]:
transition_matrix = torch.zeros((l, l))

In [7]:
# the following are the coordinates and values, using 1-indexing
transitions = [
    (
        (1, 2), #row, col
        0.3 #prob
    ),
    (
        (1, 3), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        1.0 #prob
    ),
    (
        (3, 4), #row, col
        1.0 #prob
    ),
    (
        (4, 5), #row, col
        1.0 #prob
    ),
    (
        (5, 6), #row, col
        0.5 #prob
    ),
    (
        (5, 7), #row, col
        0.5 #prob
    ),
    (
        (6, 9), #row, col nice
        1.0 #prob
    ),
    (
        (7, 8), #row, col
        1.0 #prob
    ),
    (
        (8, 9), #row, col
        1.0 #prob
    ),
]

In [8]:
for (row, col), prob in transitions:
    transition_matrix[row-1, col-1] = prob

In [9]:
token_probs = torch.zeros((l, vocab_size))

In [10]:
# the following are the coordinates and values, using 1-indexing
# format is
# (row, col), value
# for example (1, 2), 0.8
# means state 1 emits token 2 with probability 0.8
emission_probs = [
    (
        (1, 1), #row, col
        0.9 #prob
    ),
    (
        (1, 5), #row, col
        0.1 #prob
    ),
    (
        (2, 2), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        0.3 #prob
    ),
    (
        (3, 2), #row, col
        0.2 #prob
    ),
    (
        (3, 3), #row, col
        0.8 #prob
    ),
    (
        (4, 3), #row, col
        0.1 #prob
    ),
    (
        (4, 4), #row, col
        0.9 #prob
    ),
    (
        (5, 4), #row, col
        0.1 #prob
    ),
    (
        (5, 5), #row, col
        0.9 #prob
    ),
    (
        (6, 6), #row, col
        0.6 #prob
    ),
    (
        (6, 7), #row, col
        0.3 #prob
    ),
    (
        (6, 8), #row, col
        0.1 #prob
    ),
    (
        (7, 5), #row, col
        0.1 #prob
    ),
    (
        (7, 6), #row, col
        0.1 #prob
    ),
    (
        (7, 7), #row, col
        0.7 #prob
    ),
    (
        (7, 2), #row, col
        0.1 #prob
    ),
    (
        (8, 6), #row, col
        0.2 #prob
    ),
    (
        (8, 8), #row, col
        0.6 #prob
    ),
    (
        (8, 9), #row, col
        0.2 #prob
    ),
    (
        (9, 8), #row, col
        0.3 #prob
    ),
    (
        (9, 9), #row, col
        0.7 #prob
    )
]

In [11]:
for (row, col), prob in emission_probs:
    token_probs[row-1, col-1] = prob

In [12]:
padded_transition_matrix = torch.zeros((padded_l, padded_l))

In [13]:
padded_transition_matrix[:l, :l] = transition_matrix

In [14]:
padded_transition_matrix

tensor([[0.0000, 0.3000, 0.7000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 0.5000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,

In [15]:
padded_token_probs = torch.zeros((padded_l, vocab_size))

In [16]:
padded_token_probs[:l, :] = token_probs

In [17]:
mask = torch.tril(torch.ones((padded_l, padded_l)))

In [18]:
mask

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]])

In [19]:
# target_lens_mask = torch.arange(padded_length * factor).repeat(len(target_lens), 1) < (target_lens * 2).unsqueeze(-1)

In [20]:
# unsqueeze transition matrix, token probs, and targets to mimic batch size of 1
# padded_transition_matrix = padded_transition_matrix.unsqueeze(0)
# padded_token_probs = padded_token_probs.unsqueeze(0)
# target = target.unsqueeze(0)
batched_transition_matrix = torch.zeros((2, padded_l, padded_l))
batched_transition_matrix[0] = padded_transition_matrix
batched_transition_matrix[1] = padded_transition_matrix

batched_token_probs = torch.zeros((2, padded_l, vocab_size))
batched_token_probs[0] = padded_token_probs
batched_token_probs[1] = padded_token_probs

batched_target = torch.zeros((2, target.shape[-1])).type(target.dtype)
batched_target[0] = target
batched_target[1] = target

In [21]:
batched_transition_matrix[0]

tensor([[0.0000, 0.3000, 0.7000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 0.5000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,

In [22]:
target_lens = torch.tensor([m, m])

In [23]:
# in this case, because we hard coded the lengths, we don't multiply by factor
target_lens_mask = torch.arange(padded_l).repeat(len(target_lens), 1) < target_lens.unsqueeze(-1)

In [24]:
target_lens_mask

tensor([[ True,  True,  True,  True,  True,  True,  True, False, False, False,
         False, False],
        [ True,  True,  True,  True,  True,  True,  True, False, False, False,
         False, False]])

In [25]:
target_lens_mask.shape

torch.Size([2, 12])

In [26]:
vertex_lens = torch.tensor([l, l])

In [27]:
vertex_lens_mask = torch.arange(padded_l).repeat(len(vertex_lens), 1) < vertex_lens.unsqueeze(-1)

In [28]:
batched_transition_matrix = torch.log(batched_transition_matrix)
batched_token_probs = torch.log(batched_token_probs)

In [29]:
target_lens_mask.shape

torch.Size([2, 12])

In [30]:
batched_transition_matrix[0]

tensor([[   -inf, -1.2040, -0.3567,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,  0.0000,    -inf,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,  0.0000,    -inf,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,  0.0000,    -inf,    -inf,    -inf,
            -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf, -0.6931, -0.6931,    -inf,
            -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
          0.0000,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,  0.0000,
            -inf,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
          0.0000,    -inf,    

In [31]:
batched_transition_mask = torch.ones_like(batched_transition_matrix)

In [32]:
vertex_lens_mask.shape

torch.Size([2, 12])

In [33]:
target_lens_mask.shape

torch.Size([2, 12])

In [34]:
batched_transition_mask.transpose(1,2)[vertex_lens_mask] = 0

In [35]:
batched_transition_mask

tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 

In [36]:
batched_transition_mask.shape # torch.Size([2, 12, 12]), int of 1s where we want to mask and 0s where we want to keep

torch.Size([2, 12, 12])

In [37]:
mask.shape # torch.Size([12, 12]), int of 1s where we want to mask and 0s where we want to keep

torch.Size([12, 12])

In [38]:
# combine mask with batched_transition_mask
batched_transition_mask = batched_transition_mask + mask


In [39]:
batched_transition_mask

tensor([[[1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [1., 1., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [1., 1., 1., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [1., 1., 1., 1., 0., 0., 0., 0., 0., 1., 1., 1.],
         [1., 1., 1., 1., 1., 0., 0., 0., 0., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 0., 0., 0., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 0., 0., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 2., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 2., 2.]],

        [[1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [1., 1., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [1., 1., 1., 0., 0., 0., 0., 0., 0., 1., 1., 1.],
         [1., 1., 1., 1., 0., 0., 0., 0., 0., 1., 1., 1.],
         [1., 1., 1., 1., 1., 0., 0., 0., 0., 1., 1., 

In [40]:
# batched_token_probs[:, ~vertex_lens_mask] = float('-inf')

In [41]:
batched_transition_matrix = fix_probs(batched_transition_matrix, batched_transition_mask)

In [42]:
batched_target.dtype

torch.int64

In [43]:
batched_transition_matrix

tensor([[[   -inf, -1.2040, -0.3567,    -inf,    -inf,    -inf,    -inf,
             -inf,    -inf,    -inf,    -inf,    -inf],
         [   -inf,    -inf,  0.0000,    -inf,    -inf,    -inf,    -inf,
             -inf,    -inf,    -inf,    -inf,    -inf],
         [   -inf,    -inf,    -inf,  0.0000,    -inf,    -inf,    -inf,
             -inf,    -inf,    -inf,    -inf,    -inf],
         [   -inf,    -inf,    -inf,    -inf,  0.0000,    -inf,    -inf,
             -inf,    -inf,    -inf,    -inf,    -inf],
         [   -inf,    -inf,    -inf,    -inf,    -inf, -0.6931, -0.6931,
             -inf,    -inf,    -inf,    -inf,    -inf],
         [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf,  0.0000,    -inf,    -inf,    -inf],
         [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
           0.0000,    -inf,    -inf,    -inf,    -inf],
         [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,
             -inf,  0.00

In [44]:
dp = dag_loss(batched_target, batched_transition_matrix, batched_token_probs)

In [45]:
dp

tensor([[[ -0.1054,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
              -inf,     -inf,     -inf,     -inf,     -inf],
         [    -inf,  -1.6660,  -2.0715,     -inf,     -inf,     -inf,     -inf,
              -inf,     -inf,     -inf,     -inf,     -inf],
         [    -inf,     -inf,  -1.8892,  -4.3741,     -inf,     -inf,     -inf,
              -inf,     -inf,     -inf,     -inf,     -inf],
         [    -inf,     -inf,     -inf,  -1.9945,  -6.6766,     -inf,     -inf,
              -inf,     -inf,     -inf,     -inf,     -inf],
         [    -inf,     -inf,     -inf,     -inf,  -2.0999,     -inf,  -9.6724,
              -inf,     -inf,     -inf,     -inf,     -inf],
         [    -inf,     -inf,     -inf,     -inf,     -inf,  -3.3038,  -5.0956,
          -11.2818,     -inf,     -inf,     -inf,     -inf],
         [    -inf,     -inf,     -inf,     -inf,     -inf,     -inf,     -inf,
           -6.7050,  -3.6602,     -inf,     -inf,     -inf],
         [   

In [47]:
dp_values = vector_gather(dp, target_lens - 1)

In [48]:
dp_values

tensor([[   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf, -6.7050,
         -3.6602,    -inf,    -inf,    -inf],
        [   -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf, -6.7050,
         -3.6602,    -inf,    -inf,    -inf]])

In [49]:
values = torch.gather(dp_values, dim=1, index=(vertex_lens - 1).unsqueeze(-1))

In [50]:
values

tensor([[-3.6602],
        [-3.6602]])